In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

User Input
   ↓
RunnableWithMessageHistory
   ↓
Load Session Memory
   ↓
Inject into Prompt
   ↓
LLM
   ↓
Save Response into Memory

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

llm = ChatOpenAI(model="gpt-5.4-nano")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

response = conversation.invoke(
    {"input": "Hi, I am Vijay"},
    config={"configurable": {"session_id": "user1"}}
)

print(response.content)

response = conversation.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user1"}}
)

print(response.content)

Hi Vijay! Nice to meet you. 😊  
How can I help you today?
Your name is **Vijay**.


In [15]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

llm = ChatOpenAI(model="gpt-5.4-nano")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    history = store[session_id]

    # Keep only last 4 messages
    history.messages = history.messages[-1:]

    return history

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

response = conversation.invoke(
    {"input": "Hi, I am Vijay"},
    config={"configurable": {"session_id": "user1"}}
)

print(response.content)

response = conversation.invoke(
    {"input": "I love cats"},
    config={"configurable": {"session_id": "user1"}}
)
print(response.content)

response = conversation.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user1"}}
)

print(response.content)

Hi Vijay! Nice to meet you 😊  
How can I help you today?
Aww cats are the best 😻 What kind of cat are you into—tiny kittens, fluffy longhairs, or sleek shorthairs? And do you have a cat yourself, or are you just a cat lover from afar?
I don’t know your name yet—what would you like me to call you?


*Summary Instead of Full History*


In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

# Store summary + recent messages
memory_store = {}

def get_memory(session_id):
    if session_id not in memory_store:
        memory_store[session_id] = {
            "summary": "",
            "recent_messages": []
        }
    return memory_store[session_id]


def summarize_old_messages(session_id):
    memory = get_memory(session_id)

    old_chat = "\n".join(memory["recent_messages"])

    prompt = f"""
    Current summary:
    {memory['summary']}

    Update the summary using these new messages:
    {old_chat}

    Return concise summary.
    """

    result = llm.invoke(prompt)

    memory["summary"] = result.content
    memory["recent_messages"] = []


def chat(session_id, user_input):
    memory = get_memory(session_id)

    # Build prompt using summary + recent messages
    prompt = f"""
    Previous conversation summary:
    {memory['summary']}

    Recent conversation:
    {' '.join(memory['recent_messages'])}

    User: {user_input}
    """

    response = llm.invoke(prompt)

    # Save latest turn
    memory["recent_messages"].append(f"User: {user_input}")
    memory["recent_messages"].append(f"AI: {response.content}")

    # If too many recent messages, summarize them
    if len(memory["recent_messages"]) >= 6:
        summarize_old_messages(session_id)

    return response.content


print(chat("user1", "Hi I am Vijay"))
print(chat("user1", "I work in IT"))
print(chat("user1", "I live in Chennai"))
print(chat("user1", "What is my name?"))

Hi Vijay! Nice to meet you 😊  
How can I help you today?
Hi Vijay! Nice to meet you 😊  
Great—what kind of IT work do you do (e.g., support, cloud, DevOps, security, software development)? And what would you like help with today?
Hi Vijay! Nice—Chennai 😊  
What would you like to do next? For example, are you looking for help with something related to your IT work (support/cloud/DevOps/security/software), or is there something else you want assistance with?
Your name is **Vijay**.
